# ELIOR — Qwen2.5-VL-3B Caption Indonesia (Zero-shot first)

Tujuan: ukur kualitas caption Bahasa Indonesia **tanpa training** dulu. Hemat kuota.

Alur:
1. Load Qwen2.5-VL-3B + **smoke test** (1 gambar) — kalau gagal, stop dalam menit.
2. Eval zero-shot di test set, **2 kondisi**: tanpa hint vs dengan hint label YOLO.
3. Metrik BLEU/CIDEr + sample caption.
4. Simpan hasil ke Kaggle dataset.
5. (OPSIONAL, default OFF) QLoRA fine-tune — nyalakan hanya kalau zero-shot kurang.

**Setup Kaggle:** Accelerator = GPU. Secrets: `KAGGLE_USERNAME`, `KAGGLE_KEY` (rotate dulu!), opsional `HF_TOKEN`.
Attach dataset: Flickr8k-Indonesia (`joykaihatu/image-caption-indonesia`) + YOLO weights-mu.

In [1]:
# Cell 1 — Install. Qwen2.5-VL perlu transformers terbaru + qwen-vl-utils.
import subprocess, sys
def pip(*p): subprocess.run([sys.executable,'-m','pip','install','-q',*p], check=True)
pip('-U','transformers>=4.49.0','accelerate>=0.34.0')
pip('qwen-vl-utils','pycocoevalcap','nltk','ultralytics')
import nltk; nltk.download('punkt',quiet=True); nltk.download('wordnet',quiet=True); nltk.download('punkt_tab',quiet=True)
print('install ok')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 85.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 98.1 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cudf-cu12 26.2.1 requires numba-cuda[cu12]<0.23.0,>=0.22.2, but you hav

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.3/104.3 MB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 52.2 MB/s eta 0:00:00
install ok


In [2]:
# Cell 2 — Imports + device
import os, json, random, gc, time
from pathlib import Path
from collections import Counter
import numpy as np, pandas as pd
from PIL import Image
from tqdm.auto import tqdm
import torch
MODEL_ID = 'Qwen/Qwen2.5-VL-3B-Instruct'   # alt ringan: 'Qwen/Qwen3-VL-2B-Instruct'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device', device, '| GPU', torch.cuda.get_device_name(0) if device=='cuda' else '-')

device cuda | GPU Tesla T4


In [3]:
# Cell 3 — Load Qwen + SMOKE TEST (1 gambar). Kalau error di sini, stop sebelum buang waktu.
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map='auto')
model.eval()
processor = AutoProcessor.from_pretrained(MODEL_ID)

PROMPT_ID = 'Deskripsikan gambar ini dalam satu kalimat Bahasa Indonesia yang jelas.'

def qwen_caption(image_path: str, hint: str = '', max_new_tokens: int = 64) -> str:
    instr = PROMPT_ID
    if hint:
        instr = f'Gambar ini berisi {hint}. ' + PROMPT_ID
    messages = [{'role':'user','content':[{'type':'image','image':image_path},{'type':'text','text':instr}]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(text=[text], images=image_inputs, videos=video_inputs,
                       padding=True, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    trimmed = out[:, inputs.input_ids.shape[1]:]
    return processor.batch_decode(trimmed, skip_special_tokens=True)[0].strip()

# Smoke test pakai gambar bawaan PIL (kotak) hanya untuk cek pipeline jalan
_tmp = Path('/kaggle/working/_smoke.jpg'); Image.new('RGB',(320,240),(120,120,120)).save(_tmp)
print('SMOKE:', qwen_caption(str(_tmp)))
print('Smoke test OK — pipeline jalan.')

config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

SMOKE: Maaf, saya tidak dapat melihat atau mendeskripsikan gambar karena saya adalah asisten berbasis teks. Saya hanya bisa membantu dengan informasi dan pertanyaan yang diberikan kepada saya.
Smoke test OK — pipeline jalan.


In [4]:
# Cell 4 — Paths. Sesuai dataset attached di Kaggle.
FLICKR_ROOT  = Path('/kaggle/input/datasets/joykaihatu/image-caption-indonesia')
YOLO_WEIGHTS = Path('/kaggle/input/datasets/claudiojuniarto/yolov11n-bestpt').rglob('*.pt')
YOLO_WEIGHTS = next(YOLO_WEIGHTS, None)
WORK = Path('/kaggle/working'); OUT = WORK/'outputs'; OUT.mkdir(parents=True, exist_ok=True)

def find_dir(root, names):
    for n in names:
        if (root/n).is_dir(): return root/n
    dirs=[x for x in root.rglob('*') if x.is_dir()]
    return max(dirs, key=lambda d: len(list(d.glob('*.jpg'))), default=None)
def find_file(root, pats):
    for p in pats:
        f=list(root.rglob(p))
        if f: return f[0]
    return None
IMG_DIR  = find_dir(FLICKR_ROOT, ['Images','images','Flicker8k_Dataset'])
CAP_FILE = find_file(FLICKR_ROOT, ['captions*.txt','caption*.csv','*.csv','Flickr8k.token.txt'])
print('IMG_DIR', IMG_DIR)
print('CAP_FILE', CAP_FILE)
print('YOLO', YOLO_WEIGHTS)
assert IMG_DIR and CAP_FILE, 'dataset Flickr tidak ketemu — cek FLICKR_ROOT'

IMG_DIR /kaggle/input/datasets/joykaihatu/image-caption-indonesia/Images
CAP_FILE /kaggle/input/datasets/joykaihatu/image-caption-indonesia/metadata.csv
YOLO /kaggle/input/datasets/claudiojuniarto/yolov11n-bestpt/best.pt


In [5]:
# Cell 5 — Parse caption, group per gambar, ambil TEST split (hemat: 300 gambar buat eval cepat)
def load_caps(f):
    if f.suffix.lower()=='.csv':
        df=pd.read_csv(f)
        ic=next((c for c in df.columns if 'image' in c.lower() or 'file' in c.lower()), df.columns[0])
        cc=next((c for c in df.columns if 'caption' in c.lower() or 'text' in c.lower()), df.columns[1])
        return df.rename(columns={ic:'image',cc:'caption'})[['image','caption']]
    rows=[]
    for line in open(f,encoding='utf-8'):
        p=line.strip().split('\t')
        if len(p)>=2: rows.append({'image':p[0].split('#')[0].strip(),'caption':p[1].strip()})
    return pd.DataFrame(rows)
df=load_caps(CAP_FILE)
g=df.groupby('image')['caption'].apply(list).reset_index(); g.columns=['image','captions']
exist={p.name for p in IMG_DIR.glob('*.jpg')}|{p.name for p in IMG_DIR.glob('*.png')}
g=g[g['image'].isin(exist)].reset_index(drop=True)
random.seed(42); idx=list(range(len(g))); random.shuffle(idx)
N_EVAL=300
test_df=g.iloc[idx[-N_EVAL:]].reset_index(drop=True)
print('total gambar valid', len(g), '| test eval', len(test_df))

total gambar valid 8091 | test eval 300


In [6]:
# Cell 6 — YOLO prompt cache (label objek Indonesia) untuk kondisi 'dengan hint'
from ultralytics import YOLO
COCO_ID={'person':'orang','bicycle':'sepeda','car':'mobil','motorcycle':'sepeda motor','dog':'anjing','cat':'kucing','chair':'kursi','laptop':'laptop','mouse':'tetikus','cup':'cangkir','bottle':'botol','tv':'televisi','keyboard':'keyboard','cell phone':'ponsel','book':'buku','dining table':'meja makan','person ':'orang'}
def tr(l): return COCO_ID.get(l.lower(), l)
def labels_prompt(labels):
    if not labels: return ''
    c=Counter(tr(l) for l in labels); parts=[o for o,_ in c.most_common(6)]
    if len(parts)==1: return parts[0]
    if len(parts)==2: return f'{parts[0]} dan {parts[1]}'
    return ', '.join(parts[:-1])+f', dan {parts[-1]}'
yolo=None
prompt_cache={}
if YOLO_WEIGHTS:
    yolo=YOLO(str(YOLO_WEIGHTS)); yolo.to(device)
    for nm in tqdm(test_df['image'], desc='YOLO cache'):
        p=IMG_DIR/nm
        try:
            r=yolo.predict(str(p), conf=0.25, verbose=False)
            labs=[yolo.names[int(c)] for c in r[0].boxes.cls.tolist()] if r[0].boxes else []
        except Exception: labs=[]
        prompt_cache[nm]=labels_prompt(labs)
    del yolo; gc.collect(); torch.cuda.empty_cache()
else:
    print('YOLO weights tak ada — kondisi hint dilewati')
print('contoh hint:', list(prompt_cache.items())[:3])

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


YOLO cache:   0%|          | 0/300 [00:00<?, ?it/s]

contoh hint: [('2423138514_950f79e432.jpg', 'orang'), ('247691240_3881777ab8.jpg', 'orang'), ('209605542_ca9cc52e7b.jpg', 'orang')]


In [7]:
# Cell 7 — Eval BLEU/CIDEr. 2 kondisi: tanpa hint vs dengan hint YOLO.
from pycocoevalcap.bleu.bleu import Bleu
from pycocoevalcap.cider.cider import Cider

def evaluate(use_hint: bool, label=''):
    hyp,ref,rows={}, {}, []
    for i,(_,r) in enumerate(tqdm(test_df.iterrows(), total=len(test_df), desc=label)):
        nm=r['image']; p=IMG_DIR/nm
        if not p.exists(): continue
        hint=prompt_cache.get(nm,'') if use_hint else ''
        cap=qwen_caption(str(p), hint=hint)
        hyp[str(i)]=[cap.lower()]; ref[str(i)]=[c.lower() for c in r['captions']]
        rows.append({'image':nm,'caption':cap,'hint':hint,'references':r['captions']})
    b,_=Bleu(4).compute_score(ref,hyp); c,_=Cider().compute_score(ref,hyp)
    m={'BLEU-1':b[0],'BLEU-2':b[1],'BLEU-3':b[2],'BLEU-4':b[3],'CIDEr':c}
    print(label, {k:round(v,4) for k,v in m.items()})
    return m, rows

t0=time.time()
m_base, r_base = evaluate(False, 'Qwen zero-shot (tanpa hint)')
m_yolo, r_yolo = (evaluate(True, 'Qwen + hint YOLO') if prompt_cache else (None,None))
print('eval %.1f menit'%((time.time()-t0)/60))

Qwen zero-shot (tanpa hint):   0%|          | 0/300 [00:00<?, ?it/s]

{'testlen': 6266, 'reflen': 4006, 'guess': [6266, 5966, 5666, 5366], 'correct': [2229, 745, 217, 62]}
ratio: 1.5641537693455905
Qwen zero-shot (tanpa hint) {'BLEU-1': 0.3557, 'BLEU-2': 0.2108, 'BLEU-3': 0.1194, 'BLEU-4': 0.0666, 'CIDEr': np.float64(0.1456)}


Qwen + hint YOLO:   0%|          | 0/300 [00:00<?, ?it/s]

{'testlen': 6045, 'reflen': 3997, 'guess': [6045, 5745, 5445, 5145], 'correct': [2249, 765, 242, 72]}
ratio: 1.5123842882157839
Qwen + hint YOLO {'BLEU-1': 0.372, 'BLEU-2': 0.2226, 'BLEU-3': 0.1301, 'BLEU-4': 0.0745, 'CIDEr': np.float64(0.1621)}
eval 40.6 menit


In [8]:
# Cell 8 — Sample caption (lihat kualitas asli) + simpan CSV
print('=== SAMPLE (zero-shot) ===')
for r in r_base[:8]: print(f"[{r['hint'] or '-'}] {r['caption']}")
pd.DataFrame(r_base).to_csv(OUT/'qwen_zeroshot.csv', index=False)
if r_yolo:
    pd.DataFrame(r_yolo).to_csv(OUT/'qwen_yolo_hint.csv', index=False)
rows=[{'Metric':k,'ZeroShot':m_base[k],'YOLO_hint':(m_yolo[k] if m_yolo else None)} for k in m_base]
df_metrics=pd.DataFrame(rows); df_metrics.to_csv(OUT/'qwen_metrics.csv', index=False)
print(df_metrics)

=== SAMPLE (zero-shot) ===
[-] Dua anak laki-laki bermain di pantai, salah satu mereka memegang tangan kanan dan tersenyum, sedangkan yang lain berdiri dengan posisi tubuh yang lebih terbuka.
[-] Seorang pria berdiri di atas batu besar, menggenggam tali belokan dan mengejar ke tingkat yang lebih tinggi.
[-] Gambar ini menunjukkan seseorang yang sedang berjalan di atas batu besar dengan tangan mereka yang terlihat di depan dan kaki mereka yang terlihat di belakang, di luar ruangan dengan pemandangan hutan di latar belakang.
[-] Dalam foto tersebut, tiga orang berpose untuk foto di sebuah acara malam, dua dari mereka memiliki penampilan yang menarik dengan riasan mata dan warna rambut yang mencolok, sementara yang ketiga mengenakan jaket kulit dan berpose dengan sen
[-] Dalam gambar tersebut, sebuah anjing berwarna putih dengan garis hitam dan cokelat sedang berlari di tepi pantai dengan laut sebagai latar belakangnya.
[-] Dalam gambar tersebut, seekor anjing hitam sedang berjalan di tep

In [11]:
# Cell 9 — Simpan hasil ke Kaggle dataset + zip + FileLink (key dari Secrets — JANGAN hardcode)
from kaggle_secrets import UserSecretsClient
import subprocess, zipfile
from IPython.display import FileLink, display
sec=UserSecretsClient()
KU=sec.get_secret('KAGGLE_USERNAME'); KK=sec.get_secret('KAGGLE_KEY')
os.makedirs('/root/.kaggle', exist_ok=True)
json.dump({'username':KU,'key':KK}, open('/root/.kaggle/kaggle.json','w')); os.chmod('/root/.kaggle/kaggle.json',0o600)
meta={'title':'elior-qwen-caption-eval','id':f'{KU}/elior-qwen-caption-eval','licenses':[{'name':'CC0-1.0'}]}
json.dump(meta, open(OUT/'dataset-metadata.json','w'))
r=subprocess.run(['kaggle','datasets','create','-p',str(OUT)], capture_output=True, text=True)
print(r.stdout, r.stderr)
if 'already exists' in (r.stdout+r.stderr).lower():
    r2=subprocess.run(['kaggle','datasets','version','-p',str(OUT),'-m','update'], capture_output=True, text=True)
    print('UPDATE:', r2.stdout, r2.stderr)
# Fallback: zip semua output working + FileLink (Save Version -> Output tab)
zp=WORK/'qwen_all_output.zip'
with zipfile.ZipFile(zp,'w',zipfile.ZIP_DEFLATED) as zf:
    for f in OUT.rglob('*'):
        if f.is_file(): zf.write(f, f.relative_to(OUT))
print('ZIP %.1f MB'%(zp.stat().st_size/1e6)); display(FileLink(str(zp)))

Starting upload for file qwen_metrics.csv
Upload successful: qwen_metrics.csv (256B)
Starting upload for file qwen_zeroshot.csv
Upload successful: qwen_zeroshot.csv (153KB)
Starting upload for file qwen_yolo_hint.csv
Upload successful: qwen_yolo_hint.csv (156KB)
Your private Dataset is being created. Please check progress at https://www.kaggle.com/datasets/claudiojuniarto/elior-qwen-caption-eval
 
  0%|          | 0.00/256 [00:00<?, ?B/s]
100%|██████████| 256/256 [00:00<00:00, 630B/s]

  0%|          | 0.00/153k [00:00<?, ?B/s]
100%|██████████| 153k/153k [00:00<00:00, 373kB/s]

  0%|          | 0.00/156k [00:00<?, ?B/s]
100%|██████████| 156k/156k [00:00<00:00, 372kB/s]

ZIP 0.1 MB


/kaggle/working/qwen_all_output.zip

## OPSIONAL — QLoRA fine-tune (default OFF)

Jalankan **hanya** kalau zero-shot di atas kurang bagus (BLEU/CIDEr rendah ATAU caption tak relevan di foto domain-mu).
QLoRA: base 4-bit + LoRA adapter. Lalu **merge ke fp16** untuk deploy ZeroGPU.

VERIFY sebelum jalan: butuh dataset training format chat Qwen. Ini kerangka — sesuaikan.

In [12]:
# Cell 10 (OPSIONAL) — QLoRA. Set RUN_FINETUNE=True untuk aktif.
RUN_FINETUNE = False
if RUN_FINETUNE:
    from peft import LoraConfig, get_peft_model
    from transformers import BitsAndBytesConfig
    bnb=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.float16)
    base=Qwen2_5_VLForConditionalGeneration.from_pretrained(MODEL_ID, quantization_config=bnb, device_map='auto')
    lora=LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias='none',
                    target_modules=['q_proj','k_proj','v_proj','o_proj'])
    base=get_peft_model(base, lora); base.print_trainable_parameters()
    print('TODO: bangun DataLoader chat-format + training loop. Lihat guide TRL SFTTrainer.')
    print('Setelah latih: simpan adapter, lalu MERGE ke fp16:')
    print("  merged = PeftModel.from_pretrained(fp16_base, adapter).merge_and_unload(); merged.save_pretrained('qwen-elior-fp16')")
else:
    print('QLoRA OFF. Nyalakan RUN_FINETUNE=True kalau zero-shot kurang.')

QLoRA OFF. Nyalakan RUN_FINETUNE=True kalau zero-shot kurang.
